In [6]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import seaborn as sns

from src.config import TRAIN_PATH, TEST_PATH
from src.preprocessing import HousePricesPreprocessor

In [7]:
pd.set_option('display.max_columns', None)

In [8]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

In [9]:
print(f"Train: {train.shape}")
print(f"Test: {test.shape}")

Train: (1460, 81)
Test: (1459, 80)


In [10]:
y = train['SalePrice']
test_ids = test['Id']

train = train.drop(['SalePrice', 'Id'], axis=1)
test = test.drop('Id', axis=1)

preprocessor = HousePricesPreprocessor()
train_processed = preprocessor.fit_transform(train, y)
test_processed = preprocessor.transform(test)

# Удаляем выбросы (только train)
outlier_idx = train_processed[
    (train_processed['GrLivArea'] > 4000) & (y < 200000)
].index
train_processed = train_processed.drop(outlier_idx)
y = y.drop(outlier_idx)

print(f"Train: {train_processed.shape}")
print(f"Test: {test_processed.shape}")
print(f"y: {y.shape}")
print(f"test_ids: {test_ids.shape}")

Train: (1458, 99)
Test: (1459, 99)
y: (1458,)
test_ids: (1459,)


In [26]:
train_missing = train.isnull().sum()
train_missing = train_missing[train_missing > 0].sort_values(ascending=False)
train_missing

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtFinType1      37
BsmtCond          37
BsmtQual          37
MasVnrArea         8
Electrical         1
dtype: int64

In [27]:
test_missing = test.isnull().sum()
test_missing = test_missing[test_missing > 0].sort_values(ascending=False)
test_missing

PoolQC          1456
MiscFeature     1408
Alley           1352
Fence           1169
MasVnrType       894
FireplaceQu      730
LotFrontage      227
GarageCond        78
GarageYrBlt       78
GarageQual        78
GarageFinish      78
GarageType        76
BsmtCond          45
BsmtExposure      44
BsmtQual          44
BsmtFinType1      42
BsmtFinType2      42
MasVnrArea        15
MSZoning           4
BsmtFullBath       2
BsmtHalfBath       2
Functional         2
Utilities          2
GarageCars         1
GarageArea         1
TotalBsmtSF        1
KitchenQual        1
BsmtUnfSF          1
BsmtFinSF2         1
BsmtFinSF1         1
Exterior2nd        1
Exterior1st        1
SaleType           1
dtype: int64

In [28]:
# Тип 1: "нет объекта" — заполняем "None" для категориальных, 0 для числовых
none_cols_cat = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
                 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
                 'MasVnrType']

none_cols_num = ['GarageYrBlt', 'GarageArea', 'GarageCars',
                 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
                 'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']

train[none_cols_cat] = train[none_cols_cat].fillna('None')
train[none_cols_num] = train[none_cols_num].fillna(0)

test[none_cols_cat] = test[none_cols_cat].fillna('None')
test[none_cols_num] = test[none_cols_num].fillna(0)

# Проверяем что осталось
remaining_train = train.isnull().sum()
remaining_train = remaining_train[remaining_train > 0].sort_values(ascending=False)

remaining_test = test.isnull().sum()
remaining_test = remaining_test[remaining_test > 0].sort_values(ascending=False)

In [29]:
print("=== Осталось в TRAIN ===")
print(remaining_train)

print(f"\n=== Осталось в TEST ===")
print(remaining_test)

=== Осталось в TRAIN ===
LotFrontage    259
Electrical       1
dtype: int64

=== Осталось в TEST ===
LotFrontage    227
MSZoning         4
Utilities        2
Functional       2
Exterior1st      1
Exterior2nd      1
KitchenQual      1
SaleType         1
dtype: int64


In [30]:
# Тип 2: реальные пропуски

# Числовые — медианой (запоминаем из train)
lotfrontage_median = train['LotFrontage'].median()
train['LotFrontage'] = train['LotFrontage'].fillna(lotfrontage_median)
test['LotFrontage'] = test['LotFrontage'].fillna(lotfrontage_median)

# Категориальные — модой (запоминаем из train)
mode_cols = ['Electrical', 'MSZoning', 'Utilities', 'Functional',
             'Exterior1st', 'Exterior2nd', 'KitchenQual', 'SaleType']

for col in mode_cols:
    mode_value = train[col].mode()[0]
    train[col] = train[col].fillna(mode_value)
    test[col] = test[col].fillna(mode_value)

In [31]:
print(f"Пропуски в train: {train.isnull().sum().sum()}")
print(f"Пропуски в test: {test.isnull().sum().sum()}")

Пропуски в train: 0
Пропуски в test: 0


In [32]:
# Константные фичи — где одно и то же значение (или почти)
nunique = train.nunique()
const_features = nunique[nunique <= 1]
almost_const = nunique[nunique <= 2]

print("Константные (1 значение):")
print(const_features if len(const_features) > 0 else "  Нет")

Константные (1 значение):
  Нет


In [33]:
print("Почти константные (2 значения):")
for col in almost_const.index:
    print(f"  {col:20s} {train[col].value_counts().to_dict()}")

Почти константные (2 значения):
  Street               {'Pave': 1454, 'Grvl': 6}
  Utilities            {'AllPub': 1459, 'NoSeWa': 1}
  CentralAir           {'Y': 1365, 'N': 95}


In [34]:
ordinal_cols = {
    'ExterQual':    ['Fa', 'TA', 'Gd', 'Ex'],
    'ExterCond':    ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtQual':     ['None', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtCond':     ['None', 'Po', 'Fa', 'TA', 'Gd'],
    'BsmtExposure': ['None', 'No', 'Mn', 'Av', 'Gd'],
    'BsmtFinType1': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'BsmtFinType2': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'HeatingQC':    ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'KitchenQual':  ['Fa', 'TA', 'Gd', 'Ex'],
    'Functional':   ['Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
    'FireplaceQu':  ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageFinish': ['None', 'Unf', 'RFn', 'Fin'],
    'GarageQual':   ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageCond':   ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'PavedDrive':   ['N', 'P', 'Y'],
    'PoolQC':       ['None', 'Fa', 'Gd', 'Ex'],
    'Fence':        ['None', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv'],
    'LotShape':     ['IR3', 'IR2', 'IR1', 'Reg'],
    'LandSlope':    ['Sev', 'Mod', 'Gtl'],
    'CentralAir':   ['N', 'Y'],
}

for col, order in ordinal_cols.items():
    mapping = {val: i for i, val in enumerate(order)}
    train[col] = train[col].map(mapping)
    test[col] = test[col].map(mapping)

In [35]:
print(f"Ordinal encoded: {len(ordinal_cols)} колонок")
print("Пример — ExterQual (Fa=0, TA=1, Gd=2, Ex=3):")
print(train['ExterQual'].value_counts().sort_index())

Ordinal encoded: 20 колонок
Пример — ExterQual (Fa=0, TA=1, Gd=2, Ex=3):
ExterQual
0     14
1    906
2    488
3     52
Name: count, dtype: int64


In [36]:
remaining_cat = train.select_dtypes(include='object').columns.tolist()
print(f"Осталось номинальных: {len(remaining_cat)}")
for col in remaining_cat:
    print(f"  {col:20s} {train[col].nunique():3d} уникальных")

Осталось номинальных: 23
  MSZoning               5 уникальных
  Street                 2 уникальных
  Alley                  3 уникальных
  LandContour            4 уникальных
  Utilities              2 уникальных
  LotConfig              5 уникальных
  Neighborhood          25 уникальных
  Condition1             9 уникальных
  Condition2             8 уникальных
  BldgType               5 уникальных
  HouseStyle             8 уникальных
  RoofStyle              6 уникальных
  RoofMatl               8 уникальных
  Exterior1st           15 уникальных
  Exterior2nd           16 уникальных
  MasVnrType             4 уникальных
  Foundation             6 уникальных
  Heating                6 уникальных
  Electrical             5 уникальных
  GarageType             7 уникальных
  MiscFeature            5 уникальных
  SaleType               9 уникальных
  SaleCondition          6 уникальных


In [37]:
low_card = [col for col in remaining_cat if train[col].nunique() <= 5]
high_card = [col for col in remaining_cat if train[col].nunique() > 5]

print(f"One-hot ({len(low_card)} колонок, <= 5 категорий):")
for col in low_card:
    print(f"  {col:20s} {train[col].nunique()}")

print(f"Много категорий ({len(high_card)} колонок, > 5):")
for col in high_card:
    print(f"  {col:20s} {train[col].nunique()}")

One-hot (10 колонок, <= 5 категорий):
  MSZoning             5
  Street               2
  Alley                3
  LandContour          4
  Utilities            2
  LotConfig            5
  BldgType             5
  MasVnrType           4
  Electrical           5
  MiscFeature          5
Много категорий (13 колонок, > 5):
  Neighborhood         25
  Condition1           9
  Condition2           8
  HouseStyle           8
  RoofStyle            6
  RoofMatl             8
  Exterior1st          15
  Exterior2nd          16
  Foundation           6
  Heating              6
  GarageType           7
  SaleType             9
  SaleCondition        6


In [38]:
train_encoded = pd.get_dummies(train, columns=low_card, drop_first=True)
test_encoded = pd.get_dummies(test, columns=low_card, drop_first=True)

In [39]:
print(f"Train до: {train.shape[1]} колонок")
print(f"Train после one-hot: {train_encoded.shape[1]} колонок")
print(f"Добавилось: {train_encoded.shape[1] - train.shape[1]} колонок")

Train до: 81 колонок
Train после one-hot: 101 колонок
Добавилось: 20 колонок


In [ ]:
for col in high_card:
    means = train.groupby(col)['SalePrice'].mean()
    train_encoded[col] = train[col].map(means)
    test_encoded[col] = test[col].map(means)

# Проверяем
print(f"Train после target encoding: {train_encoded.shape[1]} колонок")
print("Пример — Neighborhood:")
neighborhood_means = train.groupby('Neighborhood')['SalePrice'].mean().sort_values()
print(neighborhood_means.head())
print("...")
print(neighborhood_means.tail())

# Проверяем NaN (если в test есть категория, которой нет в train)
test_nulls = test_encoded[high_card].isnull().sum()
test_nulls = test_nulls[test_nulls > 0]
if len(test_nulls) > 0:
    print("NaN в test после target encoding:")
    print(test_nulls)
else:
    print("NaN в test: нет")

Train после target encoding: 101 колонок

Пример — Neighborhood:
Neighborhood
MeadowV     98576.470588
IDOTRR     100123.783784
BrDale     104493.750000
BrkSide    124834.051724
Edwards    128219.700000
Name: SalePrice, dtype: float64
...
Neighborhood
Veenker    238772.727273
Timber     242247.447368
StoneBr    310499.000000
NridgHt    316270.623377
NoRidge    335295.317073
Name: SalePrice, dtype: float64

NaN в test: нет


In [41]:
print("GrLivArea выбросы (>4000 кв.футов):")
print(train[train['GrLivArea'] > 4000][['GrLivArea', 'SalePrice']])

print("TotalBsmtSF выбросы (>5000):")
print(train[train['TotalBsmtSF'] > 5000][['TotalBsmtSF', 'SalePrice']])

print("LotArea выбросы (>100000):")
print(train[train['LotArea'] > 100000][['LotArea', 'SalePrice']])

GrLivArea выбросы (>4000 кв.футов):
      GrLivArea  SalePrice
523        4676     184750
691        4316     755000
1182       4476     745000
1298       5642     160000
TotalBsmtSF выбросы (>5000):
      TotalBsmtSF  SalePrice
1298         6110     160000
LotArea выбросы (>100000):
     LotArea  SalePrice
249   159000     277000
313   215245     375000
335   164660     228950
706   115149     302000


In [42]:
print(f"Train до удаления: {train_encoded.shape[0]}")

outlier_idx = train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 200000)].index
train_encoded = train_encoded.drop(outlier_idx)
train = train.drop(outlier_idx)

print(f"Train после удаления: {train_encoded.shape[0]}")
print(f"Удалено строк: {len(outlier_idx)}")

Train до удаления: 1460
Train после удаления: 1458
Удалено строк: 2
